# HealthConnect Week 4: Machine Learning Problem Definition

**AnalystLab Africa - Data Science Internship Programme**  
**Intern:** Wilson Moses | **Batch:** D  
**Project:** Improving Patient Appointment Attendance and Healthcare Support Using Data and AI

This notebook applies Business Understanding, Analytical Approach, Data Requirements, Data Collection and initial Data Understanding from the IBM Data Science Methodology.

## Business Understanding

- **Business problem:** Missed appointments may waste capacity and disrupt operations.
- **Data Science problem:** Determine whether pre-appointment information can predict no-show risk.
- **Possible action:** Prioritise appropriate confirmation, reminder or rescheduling support.
- **Boundary:** Prediction creates value only when connected to a defined and evaluated intervention.

## Analytical Approach

The initial task is supervised binary classification at appointment level. Only information available before the selected prediction time should be eligible as an input feature.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_paths = [
    Path('../data/raw/healthconnect_appointment_data.csv'),
    Path('data/raw/healthconnect_appointment_data.csv'),
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place healthconnect_appointment_data.csv in data/raw.')
df_raw = pd.read_csv(DATA_PATH, keep_default_na=False)
df = df_raw.copy()
print(df.shape)

(5000, 18)


## Data Requirements and Collection

The solution requires labelled appointment history, a stable prediction time and features available before that time. Reminder and cancellation timestamps remain collection gaps.

In [2]:
overview = pd.DataFrame({
    'Measure': ['Rows', 'Columns', 'Unique appointment IDs', 'Unique patient IDs'],
    'Value': [len(df), df.shape[1], df['appointment_id'].nunique(), df['patient_id'].nunique()]
})
print(overview.to_string(index=False))

               Measure  Value
                  Rows   5000
               Columns     18
Unique appointment IDs   5000
    Unique patient IDs   1696


In [3]:
for col in ['booking_date', 'appointment_date']:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')
for col in ['distance_to_clinic_km', 'waiting_time_minutes']:
    df[col] = pd.to_numeric(df[col].replace('', np.nan), errors='coerce')
print(df[['booking_date', 'appointment_date']].agg(['min', 'max']).to_string())

    booking_date appointment_date
min   2024-11-07       2025-01-01
max   2026-06-27       2026-06-30


## Initial Data Understanding

The following checks assess structure, missingness, uniqueness, date consistency and operational plausibility.

In [4]:
quality_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_count': df.nunique(dropna=False),
}).reset_index(names='variable')
print(quality_summary.to_string(index=False))

             variable   dtype  missing_count  missing_pct  unique_count
       appointment_id  object              0          0.0          5000
           patient_id  object              0          0.0          1696
               gender  object              0          0.0             3
                  age   int64              0          0.0            63
            age_group  object              0          0.0             6
     appointment_type  object              0          0.0             4
         booking_date  object              0          0.0           593
     appointment_date  object              0          0.0           545
      appointment_day  object              0          0.0             7
     appointment_time  object              0          0.0             3
    booking_lead_days   int64              0          0.0            61
previous_appointments   int64              0          0.0            12
    previous_no_shows   int64              0          0.0       

In [5]:
checks = pd.Series({
    'exact_duplicate_rows': df.duplicated().sum(),
    'duplicate_appointment_ids': df['appointment_id'].duplicated().sum(),
    'sunday_appointments': df['appointment_date'].dt.dayofweek.eq(6).sum(),
})
print(checks.to_string())

exact_duplicate_rows           0
duplicate_appointment_ids      0
sunday_appointments          737


In [6]:
print(pd.crosstab(df['reminder_sent'], df['reminder_channel'], dropna=False).to_string())

reminder_channel  Email  None   SMS  WhatsApp
reminder_sent                                
No                    0  1366     0         0
Yes                 533     0  2000      1101


In [7]:
outcomes = df['appointment_outcome'].value_counts().rename('count').to_frame()
outcomes['share_pct'] = (outcomes['count'] / len(df) * 100).round(2)
print(outcomes.to_string())

                     count  share_pct
appointment_outcome                  
No-Show               2423      48.46
Attended              2314      46.28
Cancelled              263       5.26


### Initial interpretation

- No exact duplicate rows or duplicate appointment identifiers were found.
- Distance is missing in 90 records and waiting time in 60.
- Reminder channel None consistently means that no reminder was sent.
- Sunday appointments conflict with the knowledge base.
- Repeated patient identifiers require cautious longitudinal interpretation.

The data supports a controlled educational prototype, subject to documented limitations.

## Target and Cancellation Handling

For the initial experiment, No-Show becomes 1 and Attended becomes 0. Cancelled remains in raw data but is excluded from the binary cohort.

In [8]:
binary_df = df[df['appointment_outcome'].isin(['Attended', 'No-Show'])].copy()
binary_df['no_show_flag'] = binary_df['appointment_outcome'].eq('No-Show').astype('int8')
summary = pd.Series({
    'binary_cohort_records': len(binary_df),
    'no_shows': int(binary_df['no_show_flag'].sum()),
    'attended': int((binary_df['no_show_flag'] == 0).sum()),
    'no_show_rate_pct': round(binary_df['no_show_flag'].mean() * 100, 2),
})
print(summary.to_string())

binary_cohort_records    4737.00
no_shows                 2423.00
attended                 2314.00
no_show_rate_pct           51.15


## Potential Features

Candidates include age, gender, appointment details, booking lead days, previous appointments, previous no-shows, reminder information, distance and waiting time. Identifiers, the source outcome and redundant features are excluded or deferred.

## Modelling and Evaluation Plan

1. Preserve raw data and document transformations.
2. Use a future-oriented time split.
3. Fit preprocessing only on training data.
4. Establish a baseline.
5. Train logistic regression first.
6. Compare selected tree-based models if justified.
7. Evaluate recall, precision, F1, confusion matrix, ranking, calibration and subgroup errors.

## Assumptions, Limitations, Risks and Dependencies

The main considerations are synthetic data, missing timing information, cancellation handling, leakage, inconsistent patient histories, fairness and the need for an operational intervention.

## Week 5 Focus

Conduct structured exploratory analysis, resolve quality issues, confirm prediction timing and create a documented modelling cohort before training models.